# AIDA 1141 Final Project – 04 Errors, Groups, Ties and the Review Policy

**Data version 2** (`shared/synthetic-data/`), using notebook 02's saved validation predictions. Nothing is refit or re-tuned.

Notebook 03 found that neither the days-until-start rule nor the logistic regression consistently does better on validation. This notebook finishes the validation work before the single test evaluation:

1. revisits how **tied scores** are handled in the matched-volume comparison. That comparison split a group of three equal-scoring assessments; the threshold-based flagging policies kept the group together;
2. explains what a flag means in a coordinator's terms, including the workload of flagging about 47% of early-warning assessments;
3. reviews the **missed at-risk assessments** and the **flags on assessments that were not at risk**, using only information available at each assessment;
4. reports results **by group**, with their sizes, and reviews bias and privacy;
5. applies development criteria fixed before this notebook's comparison, then **chooses and freezes the review policy**.

Everything uses **validation** data. The test set and the time-based check stay unopened until the policy is frozen. The simulation's evaluation-only values are not used. All records are synthetic, and the rates describe the generator's assumptions, not real employees or real HR performance.

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_VERSION = "2"
SD = Path("../../../shared/synthetic-data")          # data version 2
OUTPUT_DIR = Path("../outputs")

settings = json.loads((OUTPUT_DIR / "model_settings.json").read_text())          # from notebook 02
predictions = pd.read_csv(OUTPUT_DIR / "validation_predictions.csv")             # from notebook 02
read = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False)
assessments = read(SD / "requests/data/assessments.csv")
requests = read(SD / "cleaning/output/requests_cleaned.csv")
employee_split = read(OUTPUT_DIR / "employee_split.csv")

sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
SOURCES = {"requests_cleaned.csv": SD / "cleaning/output/requests_cleaned.csv",
           "assessments.csv": SD / "requests/data/assessments.csv",
           "employee_split.csv": OUTPUT_DIR / "employee_split.csv", "feature_list.csv": OUTPUT_DIR / "feature_list.csv"}
assert {n: sha(p) for n, p in SOURCES.items()} == settings["source_fingerprints_sha256"], \
    "The inputs don't match the fingerprints saved with the model in notebook 02"

# Eligible validation assessments rebuilt from the source files with notebook 01's rules
source = assessments.merge(employee_split[["employee_id", "split"]], on="employee_id", how="left", validate="many_to_one")
eligible_ids = set(source.loc[(source["split"] == "validation") & (source["at_risk"] != "")
                              & (source["past_deadline_at_assessment"] == "False"), "assessment_id"])
assert set(predictions["assessment_id"]) == eligible_ids and predictions["assessment_id"].is_unique, \
    "The predictions are not exactly the eligible validation assessments"

# The saved rows must also carry the source's task, employee and outcome label (label normalised to 0/1)
saved = predictions.set_index("assessment_id")[["request_id", "employee_id", "at_risk"]]
truth = source.set_index("assessment_id").loc[saved.index, ["request_id", "employee_id", "at_risk"]]
assert truth["at_risk"].isin(["True", "False"]).all() and saved["at_risk"].isin([0, 1]).all(), "Unexpected label values"
assert (saved["request_id"] == truth["request_id"]).all() and (saved["employee_id"] == truth["employee_id"]).all(), \
    "Saved task or employee IDs don't match the source assessments"
assert (saved["at_risk"] == truth["at_risk"].map({"True": 1, "False": 0})).all(), \
    "Saved outcome labels don't match the source assessments"

# Add the remaining model inputs (observable at each assessment) for the error and group review
FEATURES = settings["features"]
v = predictions.merge(assessments[["assessment_id", "current_status", "business_days_in_current_status", "team_open_workload"]],
                      on="assessment_id", how="left", validate="one_to_one")
v = v.merge(requests[["request_id", "hiring_department", "intake_channel", "info_issue_at_intake"]],
            on="request_id", how="left", validate="many_to_one")
v["intake_channel"] = v["intake_channel"].replace("", "unknown")              # as in notebook 02
for c in ["business_days_in_current_status", "team_open_workload"]:
    v[c] = v[c].astype(int)
v["y"] = v["at_risk"].astype(int)
assert len(v) == len(predictions) and v[FEATURES].notna().all().all(), "Some model inputs are missing after the merge"
assert not (v[FEATURES].astype(str).apply(lambda c: c.str.strip()) == "").any().any(), "Some model inputs are blank"

DAYS_THRESHOLD = settings["days_threshold"]
MODEL_THRESHOLD = settings["model_threshold"]
assert (v["days_rule"] == (v["business_days_until_start"] <= DAYS_THRESHOLD)).all()
assert (v["logistic_regression"] == (v["risk_probability"] >= MODEL_THRESHOLD)).all()
METHODS = {"days rule": "days_rule", "logistic regression": "logistic_regression"}

print(f"Data version {DATA_VERSION}: {len(v):,} validation early-warning assessments, {v['y'].sum()} at risk, "
      f"{v.loc[v['y'] == 1, 'request_id'].nunique()} at-risk tasks, {v['employee_id'].nunique()} employees")
print(f"Training-selected thresholds from notebook 02: days until start <= {DAYS_THRESHOLD}; "
      f"model risk >= {MODEL_THRESHOLD:.4f}")

Data version 2: 1,033 validation early-warning assessments, 194 at risk, 154 at-risk tasks, 209 employees
Training-selected thresholds from notebook 02: days until start <= 8; model risk >= 0.1338


## A. Shared measures

The same measures are used throughout, at two levels:

- **Assessment recall:** of the at-risk assessments, how many received a warning?
- **Task recall:** of the at-risk tasks, how many received at least one early-warning flag? A task can be assessed twice (checkpoints 1 and 2), so a task counts as flagged if any of its assessments was flagged. This is **task deduplication**; in a live queue, a later assessment of a task already on the list would update its entry rather than create a second one.
- **Precision:** of the flagged assessments, how many were labelled at risk? Its denominator is the number flagged.
- **Flags on not-at-risk assessments:** flags on assessments labelled not at risk (the task was finished in time). They count toward review volume, but they do not tell us whether a check would have been useful or how much staff time it would have taken.
- **Unique tasks flagged:** the number of different tasks flagged across the observed checkpoints. It shows how much the assessment counts shrink after deduplication; it does not measure the size of a live queue at any one time.

Counts are whole numbers; rates and percentages may have decimals. The function below only accepts real True/False (or 0/1) flags. The fractional expected results for tied orders in section B are calculated separately: they are analytical estimates, not sets of flags.

In [2]:
def summarize(flag, rows=None):
    # Review volume and outcomes when the given assessments are flagged
    rows = v if rows is None else rows
    if isinstance(flag, pd.Series) and not flag.index.equals(rows.index):
        raise ValueError("Flags must align with the assessment rows")
    flag = pd.Series(flag, index=rows.index)
    if not flag.isin([True, False]).all():                 # rejects missing values and strings such as "False"
        raise ValueError("Flags must contain only booleans or 0/1, with no missing values")
    flag = flag.astype(bool)
    y = rows["y"] == 1
    tp, fp, fn = int((flag & y).sum()), int((flag & ~y).sum()), int((~flag & y).sum())
    tasks = rows.assign(flag=flag).groupby("request_id").agg(at_risk=("y", "max"), flagged=("flag", "any"))
    at_risk_tasks = int(tasks["at_risk"].sum())
    caught_tasks = int((tasks["at_risk"].astype(bool) & tasks["flagged"]).sum())
    return {"assessments": len(rows), "at_risk": int(y.sum()), "flagged": int(flag.sum()),
            "at_risk_flagged": tp, "not_at_risk_flagged": fp, "at_risk_missed": fn,
            "assessment_recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "unique_tasks_flagged": int(tasks["flagged"].sum()), "at_risk_tasks": at_risk_tasks,
            "at_risk_tasks_flagged": caught_tasks, "at_risk_tasks_never_flagged": at_risk_tasks - caught_tasks,
            "task_recall": caught_tasks / at_risk_tasks if at_risk_tasks else np.nan}


COUNT_COLS = ["assessments", "at_risk", "flagged", "at_risk_flagged", "not_at_risk_flagged", "at_risk_missed",
              "unique_tasks_flagged", "at_risk_tasks", "at_risk_tasks_flagged", "at_risk_tasks_never_flagged"]
overall = pd.DataFrame({name: summarize(v[col]) for name, col in METHODS.items()}).T
overall[COUNT_COLS] = overall[COUNT_COLS].astype(int)
overall = overall.astype(object)                       # keep counts as whole numbers when rows are read back
display(overall)

,assessments,at_risk,flagged,at_risk_flagged,not_at_risk_flagged,at_risk_missed,assessment_recall,precision,unique_tasks_flagged,at_risk_tasks,at_risk_tasks_flagged,at_risk_tasks_never_flagged,task_recall
days rule,1033,194,486,177,309,17,0.912371,0.364198,366,154,154,0,1.0
logistic regression,1033,194,494,178,316,16,0.917526,0.360324,368,154,154,0,1.0
